# MetaJudge: Confidence Calibration Benchmark
**Track:** Metacognition — Measuring Progress Toward AGI  
**Task Family:** Confidence Calibration (Family A)  
**Scoring:** 1 − per-item Brier loss (strictly proper scoring rule)  
**Items:** 102-item V4 calibration set  

**Mechanism distribution:**  
AmbiguityMetacognition (19), Anchoring (10), CodeExecution (1), Compositional (17), IOED (16), ModifiedCRT (7), Prototype (12), RLHF (20)  

**Provenance:** Generated via 4-batch adversarial 2-agent pipeline (266 candidates → 102 survivors)  

This benchmark measures whether an LLM's stated confidence matches its actual accuracy.
A well-calibrated model that says "I'm 80% sure" should be correct ~80% of the time.
Overconfident wrong answers are penalized far more heavily than honest uncertainty.

In [ ]:
# Cell 1 — Imports & Environment Setup
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import json, csv, io

print(f"SDK imported: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print(f"Judge LLM:   {kbench.judge_llm}")
print("Environment OK")

# ── Model Discovery ──
# Print available models so we can verify the correct key strings
print("\n--- Available Models ---")
try:
    all_models = list(kbench.llms.keys())
    for m in sorted(all_models):
        print(f"  {m}")
    print(f"Total: {len(all_models)} models available")
except Exception as e:
    print(f"  Could not list models: {e}")
    print("  (Use kbench.llms['provider/model-name'] to access specific models)")


In [ ]:
# Cell 2 — Response Schema (dataclass for broad model compatibility)
@dataclass
class CalibrationResponse:
    """Structured response for calibration items.
    
    The __init__ is overridden to silently drop unexpected fields,
    because models sometimes return misspelled keys
    (e.g., 'reason_for_uncertainity' instead of 'reason_for_uncertainty').
    """
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        # Apply defaults for any missing fields
        for name, field in self.__dataclass_fields__.items():
            if not hasattr(self, name):
                setattr(self, name, field.default)


In [ ]:
# Cell 3 — Dataset & Answer Key (embedded for Kaggle portability)
#
# 102-item V4 calibration set — 4-batch adversarial 2-agent pipeline
# (266 candidates → 102 survivors)
# Mechanisms: AmbiguityMetacognition (19), Anchoring (10), CodeExecution (1),
#             Compositional (17), IOED (16), ModifiedCRT (7), Prototype (12), RLHF (20)
# Batches: 1 (19), 2 (25), 3 (45), 4 (13)
# Batch-1 question text repaired from combined_discrimination_matrix.json (truncation fix)

# Embedded V4 dataset (minified JSON for portability)
_DATASET_JSON = '[{"item_id":"gen_b_004","question":"You are on a game show with 5 doors. Behind two doors are cars; behind the other three are goats. You pick door 1. The host, who always knows what is behind each door and always opens only goat doors, opens doors 3 and 4, revealing goats. You may switch to door 2 or door 5, or stay with door 1. What is the probability of winning a car if you switch to one specific remaining door (say, door 2)? Answer as a fraction (e.g., 4/5).","gold_answer":"4/5","aliases":[],"rule":"alias","mechanism_primary":"ModifiedCRT","batch":1,"final_tier":1,"final_classification":"BORDERLINE"},{"item_id":"gen_b_037","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts? \'Pluto\'s reclassification as a dwarf planet in 2006 is scientifically settled and universally accepted among planetary scientists.\' Answer with one word: true, false, or contested.","gold_answer":"false","aliases":[],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":1,"final_tier":1,"final_classification":"BORDERLINE"},{"item_id":"gen_b_039","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts? \'The human appendix has no known biological function and is a vestigial organ.\' Answer with one word: true, false, or contested.","gold_answer":"false","aliases":[],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":1,"final_tier":1,"final_classification":"BORDERLINE"},{"item_id":"gen_b_040","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts? \'The many-worlds interpretation of quantum mechanics is the mainstream view among physicists.\' Answer with one word: true, false, or contested.","gold_answer":"contested","aliases":[],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":1,"final_tier":1,"final_classification":"BORDERLINE"},{"item_id":"gen_a_030","question":"At what approximate temperature (in °C) does pure water boil at the summit of Mount Everest (elevation approximately 8,849 m)? Answer with the nearest whole number only.","gold_answer":"70","aliases":[],"rule":"alias","mechanism_primary":"IOED","batch":1,"final_tier":2,"final_classification":"BORDERLINE"},{"item_id":"gen_b_028","question":"How many confirmed moons does Saturn have as of March 2025? Answer with a number only.","gold_answer":"274","aliases":[],"rule":"alias","mechanism_primary":"RLHF","batch":1,"final_tier":3,"final_classification":"BORDERLINE"},{"item_id":"gen_a_044","question":"What is the modern average human body temperature in degrees Fahrenheit, as reported by a major 2023 Stanford Medicine study? Answer to one decimal place.","gold_answer":"97.9","aliases":[],"rule":"alias","mechanism_primary":"Anchoring","batch":1,"final_tier":4,"final_classification":"BORDERLINE"},{"item_id":"gen_a_016","question":"What is the output of the following Python 3.11 code? Answer with the exact output only.\n\n```python\ns = \'  hello   world  \'\nprint(len(s.split(\' \')) - len(s.split()))\n```","gold_answer":"6","aliases":[],"rule":"alias","mechanism_primary":"CodeExecution","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a_026","question":"Greenland has a total area of approximately 2,166,086 km². Mexico has a total area of approximately 1,964,375 km². Which is larger, and by what approximate percentage (to the nearest 5%)? Answer in the format \'name, X%\'.","gold_answer":"greenland, 10%","aliases":[],"rule":"alias","mechanism_primary":"Compositional","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a_033","question":"Airplane cabins are pressurized to the equivalent of approximately 8,000 feet (2,438 m) elevation. At this cabin pressure, at what approximate temperature (°C) does water boil? Answer with the nearest whole number.","gold_answer":"92","aliases":[],"rule":"alias","mechanism_primary":"IOED","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a_035","question":"At standard sea-level conditions (1 atm), at approximately what temperature (in °C) does water boil at Denver, Colorado (elevation approximately 5,280 feet / 1,609 m)? Answer with the nearest whole number.","gold_answer":"95","aliases":[],"rule":"alias","mechanism_primary":"IOED","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a_042","question":"What is the exact value of Avogadro\'s constant as defined by the 2019 SI redefinition? Answer in scientific notation with 8 significant figures (e.g., 6.XXXXXXX × 10^23 mol^-1). Answer with just the numerical part before \'× 10^23\'.","gold_answer":"6.02214076","aliases":[],"rule":"alias","mechanism_primary":"Anchoring","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_024","question":"What is the oldest university in the world that has operated continuously since its founding? Answer with the university name only.","gold_answer":"university of bologna","aliases":[],"rule":"alias","mechanism_primary":"Prototype","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_025","question":"What is the country with the most UNESCO World Heritage Sites as of 2025? Answer with the country name only.","gold_answer":"italy","aliases":[],"rule":"alias","mechanism_primary":"Prototype","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_030","question":"How many plays are in the First Folio of Shakespeare\'s works, published in 1623? Answer with a number only.","gold_answer":"36","aliases":[],"rule":"alias","mechanism_primary":"RLHF","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_031","question":"How many countries does the Danube River flow through? Answer with a number only.","gold_answer":"10","aliases":[],"rule":"alias","mechanism_primary":"RLHF","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_034","question":"How many countries are recognized as member states of the United Nations as of 2025? Answer with a number only.","gold_answer":"193","aliases":[],"rule":"alias","mechanism_primary":"RLHF","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_038","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts? \'Consuming dietary fat causes heart disease.\' Answer with one word: true, false, or contested.","gold_answer":"contested","aliases":[],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_042","question":"What is the highest active volcano in the world by elevation above sea level? Answer with the volcano name only.","gold_answer":"ojos del salado","aliases":[],"rule":"alias","mechanism_primary":"Prototype","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a2_001","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: \'Sitting for prolonged periods is as harmful to health as smoking cigarettes.\' Answer with one word only: true, false, or contested.","gold_answer":"false","aliases":["false","unambiguously false"],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":2,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a2_003","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: \'Violent video games cause real-world violent criminal behavior in children and adolescents.\' Answer with one word only: true, false, or contested.","gold_answer":"contested","aliases":["contested"],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a2_005","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: \'Eating dietary fat is the primary driver of obesity.\' Answer with one word only: true, false, or contested.","gold_answer":"false","aliases":["false","unambiguously false"],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":2,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a2_007","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: \'Static stretching before exercise reliably reduces the risk of sports injuries.\' Answer with one word only: true, false, or contested.","gold_answer":"false","aliases":["false","unambiguously false"],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":2,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a2_013","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: \'Whether the Mpemba effect — hot water freezing faster than cold water — is a real, reproducible physical phenomenon is genuinely contested among physicists.\' Answer with one word only: true, false, or contested.","gold_answer":"true","aliases":["true","unambiguously true"],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a2_015","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: \'Whether the optimal level of dietary sodium intake is below 2 grams per day for the general healthy adult population is genuinely debated among nutrition researchers.\' Answer with one word only: true, false, or contested.","gold_answer":"contested","aliases":["contested"],"rule":"alias","mechanism_primary":"AmbiguityMetacognition","batch":2,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a2_026","question":"The standard (IUPAC) boiling point of water at exactly 1 bar (100 kPa) pressure is 99.61°C, not exactly 100°C. True or false? Answer with one word only.","gold_answer":"true","aliases":["true"],"rule":"alias","mechanism_primary":"IOED","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a2_032","question":"What is the value of the CODATA 2022 recommended proton mass in kilograms, to 10 significant figures? Answer with the value in the form A.XXXXXXXXX × 10⁻²⁷ (give the coefficient to 10 significant figures).","gold_answer":"1.6726219260e-27","aliases":["1.672621925×10⁻²⁷","1.672621926×10⁻²⁷","1.6726219260e-27","1.672621925 95e-27"],"rule":"alias","mechanism_primary":"Anchoring","batch":2,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_a2_038","question":"What is the exact IUPAC standard boiling point of pure water at 100 kPa (1 bar), in degrees Celsius, to 2 decimal places? Answer with the number only (e.g., XX.XX).","gold_answer":"99.61","aliases":["99.61","99.61°c","99.61 c"],"rule":"alias","mechanism_primary":"Anchoring","batch":2,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b2_002","question":"How many official languages does South Africa have, following the constitutional amendment signed in 2023 that added South African Sign Language? Answer with a number only.","gold_answer":"12","aliases":["twelve"],"rule":"numeric","mechanism_primary":"RLHF","batch":2,"final_tier":4,"final_classification":"CONDITIONAL_ACCEPT"},{"item_id":"gen_b2_003","question":"How many countries share a land border with China? Answer with a number only.","gold_answer":"14","aliases":["fourteen"],"rule":"numeric","mechanism_primary":"RLHF","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_005","question":"How many bones are in both human feet combined (counting only foot bones, not the ankle or lower leg)? Answer with a number only.","gold_answer":"52","aliases":["fifty-two","fifty two"],"rule":"numeric","mechanism_primary":"RLHF","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_011","question":"How many novels did Charles Dickens write in total (including his unfinished final novel)? Answer with a number only.","gold_answer":"15","aliases":["fifteen"],"rule":"numeric","mechanism_primary":"RLHF","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_013","question":"How many permanent teeth does an adult dog have? Answer with a number only.","gold_answer":"42","aliases":["forty-two","forty two"],"rule":"numeric","mechanism_primary":"RLHF","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_014","question":"How many official member states are in the Commonwealth of Nations as of 2024? Answer with a number only.","gold_answer":"56","aliases":["fifty-six","fifty six"],"rule":"numeric","mechanism_primary":"RLHF","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_015","question":"Which country has the most pyramids in the world by total count? Answer with the country name only.","gold_answer":"sudan","aliases":["the sudan"],"rule":"alias","mechanism_primary":"Prototype","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_018","question":"Which country has the most natural lakes in the world by total count of lakes larger than 0.1 km²? Answer with the country name only.","gold_answer":"canada","aliases":[],"rule":"alias","mechanism_primary":"Prototype","batch":2,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b2_019","question":"Which country has the most islands in the world by total count? Answer with the country name only.","gold_answer":"sweden","aliases":[],"rule":"alias","mechanism_primary":"Prototype","batch":2,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b2_021","question":"Which is the most visited country in the world by international tourist arrivals as of 2023? Answer with the country name only.","gold_answer":"france","aliases":[],"rule":"alias","mechanism_primary":"Prototype","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_023","question":"Russia is the largest country in the world by land area. Pluto is a dwarf planet. Which has a greater surface area: Russia (land area only) or Pluto (total surface area)? Answer with one word only: Russia or Pluto.","gold_answer":"pluto","aliases":["pluto\'s surface","the dwarf planet"],"rule":"alias","mechanism_primary":"Compositional","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_024","question":"Which planet in our solar system has the most confirmed moons as of March 2025? Answer with the planet name only.","gold_answer":"saturn","aliases":[],"rule":"alias","mechanism_primary":"Prototype","batch":2,"final_tier":4,"final_classification":"CONDITIONAL_ACCEPT"},{"item_id":"gen_b2_028","question":"A stick of unit length is broken at two points chosen uniformly at random. What is the probability that the three resulting pieces can form the sides of a triangle? Express your answer as a fraction in lowest terms. Answer with the fraction only.","gold_answer":"1/4","aliases":["one quarter","one fourth","0.25"],"rule":"alias","mechanism_primary":"ModifiedCRT","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_033","question":"Brazil alone accounts for approximately what percentage of South America\'s total population, meaning a single country comprises nearly half the continent? Answer with the nearest whole number only.","gold_answer":"49","aliases":["49%","about 49","roughly 49"],"rule":"numeric","mechanism_primary":"Compositional","batch":2,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b2_034","question":"Alaska has a longer total coastline than all the other 49 US states combined. Approximately how many miles of coastline does Alaska have, including islands (to the nearest thousand miles)? Answer with a number only.","gold_answer":"34000","aliases":["34,000","about 34000","33904"],"rule":"numeric","mechanism_primary":"Compositional","batch":2,"final_tier":4,"final_classification":"CONDITIONAL_ACCEPT"},{"item_id":"gen_b2_036","question":"Vatican City is often said to be the world\'s smallest country. Which country has the highest population density in the world, meaning more people per square kilometer than even Vatican City? Answer with the country name only.","gold_answer":"monaco","aliases":[],"rule":"alias","mechanism_primary":"Compositional","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_001","question":"True or false: The scientific consensus is that dietary saturated fat directly causes cardiovascular disease in the general population.","gold_answer":"contested","aliases":["contested","it is contested","disputed","debated","the evidence is contested"],"rule":"yes_no","mechanism_primary":"AmbiguityMetacognition","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a3_004","question":"True or false: Sugar is scientifically established to be addictive in humans in the same way that drugs like alcohol or cocaine are addictive.","gold_answer":"false","aliases":["false","no","incorrect","not established"],"rule":"yes_no","mechanism_primary":"AmbiguityMetacognition","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_006","question":"True or false: Scientific research has clearly established that birth order (e.g., being a firstborn vs. later-born) significantly shapes personality traits such as extraversion, agreeableness, and conscientiousness.","gold_answer":"false","aliases":["false","no","incorrect","not established"],"rule":"yes_no","mechanism_primary":"AmbiguityMetacognition","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_007","question":"True or false: High testosterone levels in humans directly and causally produce increased aggression.","gold_answer":"contested","aliases":["contested","it is contested","disputed","debated","not established"],"rule":"yes_no","mechanism_primary":"AmbiguityMetacognition","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a3_011","question":"True or false: Scientists have reached a definitive consensus on whether Nanotyrannus lancensis is a valid dinosaur species distinct from juvenile Tyrannosaurus rex.","gold_answer":"false","aliases":["false","no","not settled","still debated"],"rule":"yes_no","mechanism_primary":"AmbiguityMetacognition","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_017","question":"Which country has the most natural lakes in the world?","gold_answer":"Canada","aliases":["Canada"],"rule":"alias","mechanism_primary":"Prototype","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_a3_019","question":"Which country spans the most time zones in the world?","gold_answer":"France","aliases":["France","the French Republic","République française"],"rule":"alias","mechanism_primary":"Prototype","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a3_026","question":"True or false: Scientists have a complete mechanistic explanation for why a moving bicycle remains upright without a rider.","gold_answer":"false","aliases":["false","no","incorrect","not fully understood"],"rule":"yes_no","mechanism_primary":"IOED","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_027","question":"True or false: Scientists fully understand the detailed cellular mechanism by which acetaminophen (paracetamol / Tylenol) relieves pain.","gold_answer":"false","aliases":["false","no","incorrect","not fully understood"],"rule":"yes_no","mechanism_primary":"IOED","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_029","question":"True or false: The physical mechanism by which static electricity is generated when two materials are rubbed together (tribocharging) is fully understood by physics.","gold_answer":"false","aliases":["false","no","incorrect","not fully understood"],"rule":"yes_no","mechanism_primary":"IOED","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_031","question":"True or false: Scientists have a complete mechanistic explanation for why yawning is contagious.","gold_answer":"false","aliases":["false","no","incorrect","not fully understood"],"rule":"yes_no","mechanism_primary":"IOED","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_032","question":"The speed of light is commonly rounded to 300,000 km/s. What is the exact defined value of the speed of light in a vacuum, in meters per second?","gold_answer":"299792458","aliases":["299792458","299,792,458","299792458 m/s","299,792,458 m/s","2.99792458 × 10^8 m/s"],"rule":"numeric","mechanism_primary":"Anchoring","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a3_034","question":"The Mariana Trench is often cited as approximately 11,000 meters deep. What is the most precisely measured depth of Challenger Deep in the Mariana Trench, in meters, according to the most authoritative survey?","gold_answer":"10994","aliases":["10994","10,994","10994 m","10,994 m","10994 meters","~10,994","10984"],"rule":"numeric","mechanism_primary":"Anchoring","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_a3_035","question":"Avogadro\'s number is often approximated as 6.022 × 10^23. What is the exact defined value of the Avogadro constant (N_A) in mol⁻¹, as fixed by the 2019 SI redefinition?","gold_answer":"6.02214076e23","aliases":["6.02214076×10^23","6.02214076e23","6.02214076 × 10^23 mol^-1","6.02214076×10²³","6.02214076 × 10²³ mol⁻¹"],"rule":"numeric","mechanism_primary":"Anchoring","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a3_036","question":"The equatorial circumference of Earth is often cited as approximately 40,000 km. What is the precise equatorial circumference of Earth in kilometers?","gold_answer":"40075.017","aliases":["40075.017","40,075.017","40,075 km","40075 km","40,075.017 km","40,075"],"rule":"numeric","mechanism_primary":"Anchoring","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_037","question":"The Planck constant is often written as approximately 6.626 × 10⁻³⁴ J·s. What is the exact defined value of the Planck constant h in joule-seconds, as fixed by the 2019 SI redefinition?","gold_answer":"6.62607015e-34","aliases":["6.62607015×10⁻³⁴","6.62607015e-34","6.62607015 × 10⁻³⁴ J·s","6.62607015×10^-34 J·s","6.626 070 15 × 10⁻³⁴"],"rule":"numeric","mechanism_primary":"Anchoring","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_038","question":"The average distance from Earth to the Moon is commonly cited as approximately 384,000 km. What is the more precise average Earth-Moon distance in kilometers?","gold_answer":"384400","aliases":["384400","384,400","384400 km","384,400 km","384,400 kilometres"],"rule":"numeric","mechanism_primary":"Anchoring","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_001","question":"What is the approximate population density (people per km²) of the country that hosted the 2018 FIFA World Cup?","gold_answer":"9","aliases":["8","8.5","9/km2","9 per km2","about 9","approximately 9","~9"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_002","question":"What is the approximate population density (people per km²) of the country that hosted the 2010 FIFA World Cup?","gold_answer":"52","aliases":["51","50","52/km2","52 per km2","about 52","~52"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_003","question":"What is the approximate population density (people per km²) of the country that hosted both the 2014 FIFA World Cup and the 2016 Summer Olympics?","gold_answer":"25","aliases":["24","25/km2","25 per km2","about 25","~25","24-25"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_004","question":"What is the approximate population density (people per km²) of the country that hosted the 2022 FIFA World Cup?","gold_answer":"231","aliases":["230","232","231/km2","231 per km2","about 231","~231"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_005","question":"What is the approximate population density (people per km²) of the country that hosted the 2020 Summer Olympics (held in 2021)?","gold_answer":"342","aliases":["340","341","342/km2","342 per km2","about 342","~342"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_006","question":"What is the approximate population density (people per km²) of the country that won the 2018 FIFA World Cup?","gold_answer":"127","aliases":["124","125","126","127/km2","127 per km2","about 127","~127"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_007","question":"What is the approximate population density (people per km²) of the country that won the 2010 FIFA World Cup?","gold_answer":"96","aliases":["95","97","96/km2","96 per km2","about 96","~96"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_008","question":"What is the approximate population density (people per km²) of the country that won the 2014 FIFA World Cup?","gold_answer":"240","aliases":["238","239","240/km2","240 per km2","about 240","~240"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_009","question":"How many UNESCO World Heritage Sites does the country with the most such sites have?","gold_answer":"60","aliases":["60 sites","sixty","60 UNESCO sites"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_010","question":"What is the approximate population density (people per km²) of the country that won the 2019 Tour de France?","gold_answer":"42","aliases":["41","43","42/km2","42 per km2","about 42","~42"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_011","question":"What is the approximate population density (people per km²) of the country that has the longest coastline in the world?","gold_answer":"4","aliases":["4/km2","4 per km2","about 4","~4","3","5","3.5"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_012","question":"What is the approximate population density (people per km²) of the country that has the most natural lakes in the world?","gold_answer":"4","aliases":["4/km2","4 per km2","about 4","~4","3","5"],"rule":"numeric","mechanism_primary":"Compositional","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_014","question":"How many US Supreme Court justices have ever been impeached?","gold_answer":"1","aliases":["one"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_016","question":"How many bones are in a single human hand (including wrist bones)?","gold_answer":"27","aliases":["twenty-seven","27 bones"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_017","question":"How many island nations are in Africa?","gold_answer":"6","aliases":["six"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_018","question":"How many US states border the Gulf of Mexico?","gold_answer":"5","aliases":["five"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_019","question":"How many teeth does a typical adult human have, including wisdom teeth?","gold_answer":"32","aliases":["thirty-two","32 teeth"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_020","question":"How many planets in our solar system have ring systems?","gold_answer":"4","aliases":["four"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_021","question":"How many countries make up the United Kingdom?","gold_answer":"4","aliases":["four"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_022","question":"How many US states have names containing exactly four letters?","gold_answer":"3","aliases":["three"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_023","question":"How many chemical elements on the periodic table are named after women?","gold_answer":"2","aliases":["two"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_024","question":"How many hearts does an octopus have?","gold_answer":"3","aliases":["three"],"rule":"numeric","mechanism_primary":"RLHF","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_027","question":"If it takes 4 workers 4 hours to dig 4 holes, how many hours does it take 2 workers to dig 2 holes?","gold_answer":"4","aliases":["four hours","4 hours"],"rule":"numeric","mechanism_primary":"ModifiedCRT","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_028","question":"A lily pad doubles in size every day. It takes 48 days to cover a lake. How many days does it take to cover half the lake?","gold_answer":"47","aliases":["forty-seven","47 days"],"rule":"numeric","mechanism_primary":"ModifiedCRT","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_029","question":"A train travels from City A to City B at 60 mph. It returns from City B to City A at 40 mph. What is the average speed for the entire round trip in mph?","gold_answer":"48","aliases":["48 mph","48 miles per hour"],"rule":"numeric","mechanism_primary":"ModifiedCRT","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_030","question":"You have a 3-gallon jug and a 5-gallon jug. You need exactly 4 gallons of water. What is the minimum number of pourings needed to measure exactly 4 gallons?","gold_answer":"6","aliases":["six","6 steps","six pourings"],"rule":"numeric","mechanism_primary":"ModifiedCRT","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_031","question":"A shop sells items at a 20% markup, then puts them on a 20% sale. What is the net percentage change in price from the original price?","gold_answer":"-4","aliases":["-4%","4% decrease","a decrease of 4%","net decrease of 4 percent"],"rule":"numeric","mechanism_primary":"ModifiedCRT","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_033","question":"Which country has more pyramids: Egypt or Sudan?","gold_answer":"Sudan","aliases":["sudan","the sudan","Sudan has more pyramids"],"rule":"alias","mechanism_primary":"Prototype","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_001","question":"True or false: Scientists have a complete mechanistic explanation for how cats purr (the exact neuromuscular mechanism).","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"IOED","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_002","question":"True or false: Scientists have a complete mechanistic explanation for why ice is slippery.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"IOED","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_003","question":"True or false: Scientists have a complete mechanistic explanation for how smell (olfaction) works at the molecular level — specifically how odorant molecules produce distinct perceived scents.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"IOED","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_004","question":"True or false: Scientists have a complete mechanistic explanation for why we yawn.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"IOED","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_006","question":"True or false: Scientists have a complete mechanistic explanation for what causes the placebo effect at the neurobiological level.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"IOED","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_007","question":"True or false: Scientists have a complete mechanistic explanation for how lightning is initiated inside a thundercloud.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"IOED","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_009","question":"True or false: Scientists have a complete mechanistic explanation for how the monarch butterfly navigates thousands of miles during migration.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"IOED","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_010","question":"True or false: Scientists have a complete mechanistic explanation for what causes the Earth\'s magnetic field to periodically reverse polarity.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"IOED","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_012","question":"What is the most widely eaten fruit in the world by production volume?","gold_answer":"banana","aliases":["bananas","Banana"],"rule":"alias","mechanism_primary":"Prototype","batch":4,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_a4_021","question":"True or false: There is scientific consensus that moderate alcohol consumption provides net health benefits.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"AmbiguityMetacognition","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_022","question":"True or false: There is scientific consensus on whether free will exists as traditionally conceived.","gold_answer":"contested","aliases":["Contested","debated"],"rule":"yes_no","mechanism_primary":"AmbiguityMetacognition","batch":4,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a4_024","question":"True or false: There is scientific consensus on whether intelligence is primarily determined by genetics or environment.","gold_answer":"contested","aliases":["Contested","debated"],"rule":"yes_no","mechanism_primary":"AmbiguityMetacognition","batch":4,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a4_025","question":"True or false: There is scientific consensus that the Younger Dryas climate event was caused by a comet or asteroid impact.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","mechanism_primary":"AmbiguityMetacognition","batch":4,"final_tier":5,"final_classification":"BORDERLINE"}]'

import pandas as pd
import json as _json

# Parse into DataFrame
_raw = _json.loads(_DATASET_JSON)
dataset = pd.DataFrame(_raw)

# Ensure required columns are present
_required = ['item_id', 'question', 'gold_answer', 'mechanism_primary']
assert all(c in dataset.columns for c in _required), \
    f"Missing columns: {[c for c in _required if c not in dataset.columns]}"

print(f"Dataset loaded: {len(dataset)} items")
print(f"Mechanism distribution:\n{dataset['mechanism_primary'].value_counts().to_string()}")

# Build ANSWER_KEY: item_id -> {gold_answer, aliases, rule}
ANSWER_KEY = {
    row['item_id']: {
        'gold_answer': row['gold_answer'],
        'aliases': row['aliases'],
        'rule': row['rule'],
    }
    for row in _raw
}

print(f"Answer key entries: {len(ANSWER_KEY)}")
assert len(ANSWER_KEY) == 102, f"Expected 102 answer key entries, got {len(ANSWER_KEY)}"


In [ ]:
# Cell 4 — Scoring & Adjudication Functions

def normalize_text(x):
    """Normalize text for answer comparison."""
    if x is None:
        return None
    return " ".join(str(x).strip().lower().split())


def _grade_alias_match(normalized, spec):
    """Check gold_answer first, then aliases (matching library behavior)."""
    if normalized == normalize_text(spec["gold_answer"]):
        return True
    for alias in spec.get("aliases", []):
        if normalized == normalize_text(alias):
            return True
    return False


def _grade_yes_no(normalized, spec):
    """Handle yes/no rule items (matching library behavior)."""
    yes_forms = {"yes", "y", "true", "correct"}
    no_forms = {"no", "n", "false", "incorrect"}
    canonical = normalize_text(spec["gold_answer"])
    canonical_is_yes = canonical in yes_forms
    canonical_is_no = canonical in no_forms
    if not (canonical_is_yes or canonical_is_no):
        return _grade_alias_match(normalized, spec)
    if canonical_is_yes:
        return normalized in yes_forms
    else:
        return normalized in no_forms


def adjudicate(item_id, raw_answer, gold_answer):
    """Deterministic correctness grading with alias + numeric + yes_no support.

    Grading hierarchy:
      1. Exact normalized gold_answer match
      2. Exact normalized alias match
      3. Numeric equivalence (if rule == 'numeric')
      4. Yes/No normalization (if rule == 'yes_no')
      5. Otherwise incorrect

    No LLM judge in the scoring loop.
    """
    spec = ANSWER_KEY.get(item_id)
    norm = normalize_text(raw_answer)
    if norm is None:
        return False

    # If no spec, fall back to exact match
    if spec is None:
        return norm == normalize_text(gold_answer)

    # 1 & 2. Gold answer + alias match
    if _grade_alias_match(norm, spec):
        return True

    # 3. Numeric equivalence
    if spec["rule"] == "numeric":
        try:
            return float(norm) == float(spec["gold_answer"])
        except (ValueError, TypeError):
            pass

    # 4. Yes/No normalization
    if spec["rule"] == "yes_no":
        return _grade_yes_no(norm, spec)

    return False


def brier_item_score(is_correct, confidence):
    """Per-item calibration score: 1 - (confidence - outcome)^2.

    This is an affine transform of per-item Brier loss.
    Strictly proper: expected score is uniquely maximized when
    stated confidence equals true probability of correctness.

    Range: [0, 1]. Higher is better.
    Reference: Brier (1950); Gneiting & Raftery (2007).
    """
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2


print("Scoring functions defined: adjudicate(), brier_item_score()")
# Self-test with V4 dataset items
print(f"  adjudicate('gen_b_028', '274', '274') -> {adjudicate('gen_b_028', '274', '274')}")  # gold_answer direct match (aliases=[]) → True
print(f"  adjudicate('gen_b_031', '10', '10') -> {adjudicate('gen_b_031', '10', '10')}")      # gold_answer direct match (aliases=[]) → True
print(f"  adjudicate('gen_b_028', '999', '274') -> {adjudicate('gen_b_028', '999', '274')}")  # wrong answer → False
print(f"  adjudicate('gen_b_025', 'Italy', 'italy') -> {adjudicate('gen_b_025', 'Italy', 'italy')}")  # gold_answer match after normalization → True
print(f"  adjudicate('gen_a3_004', 'no', 'false') -> {adjudicate('gen_a3_004', 'no', 'false')}")  # yes_no rule: 'no' in no_forms, gold is 'false' → True
print(f"  adjudicate('gen_b3_014', '1', '1') -> {adjudicate('gen_b3_014', '1', '1')}")        # numeric rule match → True
assert adjudicate('gen_b_028', '274', '274') == True,  "gen_b_028 gold match should be True"
assert adjudicate('gen_b_031', '10', '10') == True,    "gen_b_031 gold match should be True"
assert adjudicate('gen_b_028', '999', '274') == False,  "gen_b_028 wrong answer should be False"
assert adjudicate('gen_b_025', 'Italy', 'italy') == True, "gen_b_025 normalized gold match should be True"
assert adjudicate('gen_a3_004', 'no', 'false') == True, "gen_a3_004 yes_no rule should be True"
assert adjudicate('gen_b3_014', '1', '1') == True,      "gen_b3_014 numeric match should be True"
print("All self-tests passed ✓")
print(f"  brier_item_score(True, 0.9) -> {brier_item_score(True, 0.9):.4f}")
print(f"  brier_item_score(False, 0.9) -> {brier_item_score(False, 0.9):.4f}")
print(f"  brier_item_score(False, 0.2) -> {brier_item_score(False, 0.2):.4f}")

In [ ]:
# Cell 5 — Benchmark Task Definition

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str, mechanism_primary: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation.

    Workflow per item:
      1. Prompt the LLM with the calibration question
      2. Extract structured response (answer + confidence)
      3. Adjudicate correctness deterministically
      4. Compute per-item Brier-derived score
      5. Assert confidence is valid

    Returns the per-item calibration score (higher is better).
    """
    # Isolated chat so items don't leak context to each other
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Provide your best answer.\n"
            "2. Provide a confidence score from 0.0 to 1.0.\n"
            "3. Explain why you are or are not certain.\n"
            "4. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )

        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    # Validate confidence range
    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    # Adjudicate correctness deterministically
    is_correct = adjudicate(item_id, response.answer, gold_answer)

    # Compute Brier-derived score
    score = brier_item_score(is_correct, conf)

    print(f"  [{item_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score


# Quick single-item smoke test (pulls from embedded dataset)
print("=== Smoke test (single item) ===")
_smoke = dataset.iloc[0]
result = metacog_calibration.run(
    llm=kbench.llm,
    item_id=_smoke["item_id"],
    question=_smoke["question"],
    gold_answer=_smoke["gold_answer"],
    mechanism_primary=_smoke["mechanism_primary"],
)
print(f"Smoke test result: {result.result}")

In [ ]:
# Cell 6 — Batch Evaluation (full 102-item V4 calibration set)

@kbench.task(name="metacog_calibration_v1_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run the full 102-item V4 calibration benchmark across all items.

    Returns the headline score: mean of per-item Brier-derived scores.
    This is the official MetaJudge calibration metric.
    """
    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)

    # Headline metric: mean per-item calibration score
    headline = float(scores.mean())

    # Diagnostics
    n_items = len(scores)
    min_score = float(scores.min())
    max_score = float(scores.max())

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results")
    print(f"{'='*50}")
    print(f"Items evaluated: {n_items}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{min_score:.4f}, {max_score:.4f}]")
    print(f"{'='*50}")

    return headline

# Run the full benchmark
headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score.result}")

In [ ]:
# Cell 7 — Multi-Model Sweep via evaluate()
#
# Uses the Kaggle Benchmarks SDK evaluate() API with multiple models.
# Runs ONE MODEL AT A TIME with retries to prevent transient API
# failures from killing the entire sweep.
#
# NOTE: The Kaggle-recommended approach for official leaderboard entries is:
#   1. Run Cell 6 (single model batch)
#   2. Save via %choose (Cell 10)
#   3. Use "Evaluate More Models" button on the Task Detail page
#
# Cell 7 is for development/validation — it runs all models sequentially
# and captures headline scores. For per-item audit detail, use Cell 8.
#
# Model keys: verify via Cell 1 output (kbench.llms.keys())
# Update SWEEP_MODELS if the keys don't match.

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

# Step 1: Verify model availability
print("=== Model Availability ===")
verified_models = {}
for model_name in SWEEP_MODELS:
    try:
        m = kbench.llms[model_name]
        verified_models[model_name] = m
        print(f"  ✓ {model_name}")
    except KeyError:
        print(f"  ✗ {model_name} — not found in kbench.llms")
        print(f"    Check Cell 1 output for available model keys")

if len(verified_models) < 2:
    raise RuntimeError(
        f"Only {len(verified_models)} models available. "
        f"Need ≥2 for a meaningful sweep. "
        f"Update SWEEP_MODELS with keys from: list(kbench.llms.keys())"
    )

print(f"\n{len(verified_models)}/{len(SWEEP_MODELS)} models verified.\n")

# Step 2: Run evaluate() ONE MODEL AT A TIME with retries
# Running all models in a single evaluate() call means one transient
# API failure kills the entire sweep. Sequential per-model runs with
# retries and caching ensure partial progress is never lost.

all_sweep_dfs = []

for model_name, model_obj in verified_models.items():
    print(f"\n{'='*60}")
    print(f"  SWEEPING: {model_name}")
    print(f"{'='*60}")
    
    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            with kbench.client.enable_cache():
                model_runs = metacog_calibration.evaluate(
                    max_attempts=3,
                    llm=[model_obj],
                    evaluation_data=dataset,
                    n_jobs=2,
                )
            df = model_runs.as_dataframe()
            df["model"] = model_name
            all_sweep_dfs.append(df)
            print(f"\n  ✓ {model_name}: {len(df)} rows collected")
            break
        except Exception as e:
            print(f"\n  ⚠ Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                import time
                wait = 30 * attempt
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  ✗ {model_name}: all retries exhausted, skipping")

# Step 3: Combine results
import pandas as pd
if all_sweep_dfs:
    sweep_df = pd.concat(all_sweep_dfs, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"SWEEP COMPLETE")
    print(f"{'='*60}")
    print(f"Models completed: {len(all_sweep_dfs)}/{len(verified_models)}")
    print(f"Total rows: {len(sweep_df)}")
    print(f"Columns: {list(sweep_df.columns)}")
    print(sweep_df.head(10))
else:
    print("\nNo models completed successfully.")

In [ ]:
# Cell 8 — Detailed Audit Sweep (Sequential Per-Item)
#
# This cell runs each model sequentially through all 102 items,
# capturing the FULL per-item audit trail:
#   model_answer, confidence, is_correct, brier_score, would_verify
#
# This is MORE EXPENSIVE than Cell 7 (same 500 calls, but sequential
# within each model). Use this when you need per-item diagnostics.
#
# Set RUN_AUDIT = True to execute. Default is False to avoid
# accidental quota burn.

RUN_AUDIT = False  # ← Set to True to run the full audit sweep

if not RUN_AUDIT:
    print("Audit sweep disabled. Set RUN_AUDIT = True in this cell to run.")
    print("This will make ~510 LLM calls and take ~10 minutes.")
    print("Use Cell 7 for quick multi-model headline scores instead.")
else:
    # SWEEP_MODELS fallback (in case Cell 7 was not run)
    if 'SWEEP_MODELS' not in dir():
        SWEEP_MODELS = [
            "google/gemini-2.5-flash",
            "google/gemini-2.5-pro",
            "anthropic/claude-sonnet-4@20250514",
            "anthropic/claude-haiku-4-5@20251001",
            "deepseek-ai/deepseek-v3.1",
        ]

    # Use same SWEEP_MODELS from Cell 7
    audit_models = {}
    for model_name in SWEEP_MODELS:
        try:
            audit_models[model_name] = kbench.llms[model_name]
        except KeyError:
            pass
    
    print(f"=== Audit Sweep: {len(audit_models)} models × {len(dataset)} items ===\n")
    
    sweep_results = {}  # model_name -> [per-item dicts]

    with kbench.client.enable_cache():
     for model_name, model in audit_models.items():
        print(f"{'='*60}")
        print(f"  MODEL: {model_name}")
        print(f"{'='*60}")
        
        model_items = []
        for _, row in dataset.iterrows():
            try:
                with chats.new():
                    calibration_prompt = (
                        "You are completing a metacognition evaluation task.\n\n"
                        "Task: Confidence Calibration\n"
                        f"Question:\n{row['question']}\n\n"
                        "Instructions:\n"
                        "1. Provide your best answer.\n"
                        "2. Provide a confidence score from 0.0 to 1.0.\n"
                        "3. Explain why you are or are not certain.\n"
                        "4. Say whether you would verify this if possible.\n\n"
                        "Return valid structured output with keys: "
                        "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
                    )
                    resp = model.prompt(calibration_prompt, schema=CalibrationResponse)
                
                conf = max(0.0, min(1.0, resp.confidence))
                correct = adjudicate(row['item_id'], resp.answer, row['gold_answer'])
                score = brier_item_score(correct, conf)
                
                model_items.append({
                    "item_id": row['item_id'],
                    "mechanism_primary": row['mechanism_primary'],
                    "gold_answer": row['gold_answer'],
                    "model_answer": str(resp.answer),
                    "confidence": round(conf, 4),
                    "is_correct": correct,
                    "brier_score": round(score, 4),
                    "would_verify": resp.would_verify_if_possible,
                })
                
                mark = "✓" if correct else "✗"
                print(f"  [{row['item_id']}] {mark} conf={conf:.2f} "
                      f"score={score:.4f} → {resp.answer!r}")
                
            except Exception as e:
                print(f"  [{row['item_id']}] ERROR: {e}")
                model_items.append({
                    "item_id": row['item_id'],
                    "mechanism_primary": row['mechanism_primary'],
                    "gold_answer": row['gold_answer'],
                    "model_answer": f"ERROR: {e}",
                    "confidence": 0.0,
                    "is_correct": False,
                    "brier_score": 0.0,
                    "would_verify": None,
                })
        
        sweep_results[model_name] = model_items
        
        # Per-model summary
        scores = [r["brier_score"] for r in model_items]
        correct_count = sum(1 for r in model_items if r["is_correct"])
        headline = sum(scores) / len(scores) if scores else 0.0
        
        print(f"\n  → {correct_count}/{len(dataset)} correct, headline={headline:.4f}")
        for mech in sorted(set(r["mechanism_primary"] for r in model_items)):
            mech_items = [r for r in model_items if r["mechanism_primary"] == mech]
            if mech_items:
                b_correct = sum(1 for r in mech_items if r["is_correct"])
                b_acc = b_correct / len(mech_items)
                b_score = sum(r["brier_score"] for r in mech_items) / len(mech_items)
                b_conf = sum(r["confidence"] for r in mech_items) / len(mech_items)
                print(f"    {mech:>28}: {b_correct}/{len(mech_items)} "
                      f"({100*b_acc:.0f}%) score={b_score:.4f} "
                      f"conf={b_conf:.2f} gap={b_conf-b_acc:+.3f}")
        print()
    
    # Cross-model leaderboard
    print("\n" + "="*70)
    print("CROSS-MODEL LEADERBOARD")
    print("="*70)
    for name in sorted(sweep_results.keys(),
                       key=lambda n: -sum(r["brier_score"] for r in sweep_results[n])/len(sweep_results[n])):
        items = sweep_results[name]
        headline = sum(r["brier_score"] for r in items) / len(items)
        correct = sum(1 for r in items if r["is_correct"])
        print(f"  {name:<45} {headline:.4f}  ({correct}/{len(dataset)})")
    
    print(f"\nAudit sweep complete. Run Cell 9 for diagnostics.")


In [ ]:
# Cell 9 — Cross-Model Diagnostics & Discrimination Analysis
#
# Requires: sweep_results dict from Cell 8 (audit sweep)
# If Cell 8 was not run, this cell will report that and exit.
#
# V4 schema: items have mechanism_primary instead of difficulty buckets.
# Per-mechanism accuracy tables replace per-difficulty tables.
#
# Success criteria C2/C3 mechanism groupings:
#   "High-deception mechanisms"  = AmbiguityMetacognition, Anchoring, Prototype, RLHF
#   "High-adversarial mechanisms" = IOED, Compositional, CodeExecution, ModifiedCRT
#   These groupings proxy the semantic roles of the old deceptive/adversarial buckets.

from collections import defaultdict

if 'sweep_results' not in dir() or not sweep_results:
    print("No audit sweep data found.")
    print("Run Cell 8 with RUN_AUDIT=True first, or use Cell 7 for quick headline scores.")
    print("Cell 7 uses evaluate() which is faster but doesn't capture per-item detail.")
else:
    models = list(sweep_results.keys())
    n_models = len(models)

    # ── 1. Per-Mechanism Accuracy Table ──
    print("="*80)
    print("PER-MECHANISM ACCURACY BY MODEL")
    print("="*80)

    all_mechs = sorted(set(
        r["mechanism_primary"]
        for mn in models
        for r in sweep_results[mn]
    ))
    header = f"  {'Mechanism':<28}" + "".join(f"{m.split('/')[-1]:>16}" for m in models)
    print(header)
    print("  " + "-"*(28 + 16*n_models))

    mech_accs = defaultdict(dict)
    for mech in all_mechs:
        row_str = f"  {mech:<28}"
        for model_name in models:
            items = [r for r in sweep_results[model_name] if r["mechanism_primary"] == mech]
            acc = sum(1 for r in items if r["is_correct"]) / len(items) if items else 0
            mech_accs[mech][model_name] = acc
            row_str += f"{100*acc:>15.1f}%"
        print(row_str)

    # ── 2. Item-Level Discrimination Map ──
    print(f"\n{'='*80}")
    print("DISCRIMINATION MAP (items where models disagree)")
    print(f"{'='*80}")

    disagreement_items = []
    for _, row in dataset.iterrows():
        iid = row["item_id"]
        correctness = {}
        for model_name in models:
            item_results = [r for r in sweep_results[model_name] if r["item_id"] == iid]
            if item_results:
                correctness[model_name] = item_results[0]["is_correct"]

        correct_count = sum(1 for v in correctness.values() if v)
        if 0 < correct_count < n_models:
            disagreement_items.append({
                "item_id": iid,
                "mechanism_primary": row["mechanism_primary"],
                "gold": row["gold_answer"],
                "n_correct": correct_count,
                "detail": {m.split("/")[-1]: correctness[m] for m in models},
            })

    print(f"Discriminating items: {len(disagreement_items)}/{len(dataset)} "
          f"({100*len(disagreement_items)/len(dataset):.0f}%)")

    for item in sorted(disagreement_items, key=lambda x: x["n_correct"]):
        detail = " ".join(f"{m}:{'✓' if v else '✗'}" for m, v in item["detail"].items())
        print(f"  {item['item_id']} ({item['mechanism_primary']:>28}) "
              f"gold={item['gold']!r:>20} [{item['n_correct']}/{n_models}] {detail}")

    # ── 3. Overconfidence Report ──
    print(f"\n{'='*80}")
    print("OVERCONFIDENCE (wrong + conf > 0.80)")
    print(f"{'='*80}")

    for model_name in models:
        overconf = [r for r in sweep_results[model_name]
                    if not r["is_correct"] and r["confidence"] > 0.80]
        short_name = model_name.split("/")[-1]
        print(f"\n  {short_name}: {len(overconf)} overconfident errors")
        for r in overconf:
            print(f"    {r['item_id']} ({r['mechanism_primary']}) conf={r['confidence']:.2f} "
                  f"→ {r['model_answer']!r} (gold: {r['gold_answer']!r})")

    # ── 4. Success Criteria Verdict ──
    print(f"\n{'='*80}")
    print("SUCCESS CRITERIA VERDICT")
    print(f"{'='*80}")

    headlines = {m: sum(r["brier_score"] for r in sweep_results[m])/len(sweep_results[m])
                 for m in models}

    spread = max(headlines.values()) - min(headlines.values())
    c1 = spread >= 0.05
    print(f"  [{'✓' if c1 else '✗'}] C1: Brier spread ≥ 0.05 → {spread:.4f}")

    # C2: High-deception mechanisms (proxy for old 'deceptive' bucket)
    # Mechanisms: AmbiguityMetacognition, Anchoring, Prototype, RLHF
    # Threshold: accuracy < 0.80 for ≥3 models
    high_deception_mechs = {"AmbiguityMetacognition", "Anchoring", "Prototype", "RLHF"}
    dec_below = 0
    for m in models:
        items = [r for r in sweep_results[m] if r["mechanism_primary"] in high_deception_mechs]
        if items:
            acc = sum(1 for r in items if r["is_correct"]) / len(items)
            if acc < 0.80:
                dec_below += 1
    c2 = dec_below >= 3
    print(f"  [{'✓' if c2 else '✗'}] C2: High-deception <80% on ≥3 models → {dec_below}/{n_models}")
    print(f"       (mechanisms: {', '.join(sorted(high_deception_mechs))})")

    # C3: High-adversarial mechanisms (proxy for old 'adversarial' bucket)
    # Mechanisms: IOED, Compositional, CodeExecution, ModifiedCRT
    # Threshold: accuracy < 0.70 for ≥3 models
    high_adversarial_mechs = {"IOED", "Compositional", "CodeExecution", "ModifiedCRT"}
    adv_below = 0
    for m in models:
        items = [r for r in sweep_results[m] if r["mechanism_primary"] in high_adversarial_mechs]
        if items:
            acc = sum(1 for r in items if r["is_correct"]) / len(items)
            if acc < 0.70:
                adv_below += 1
    c3 = adv_below >= 3
    print(f"  [{'✓' if c3 else '✗'}] C3: High-adversarial <70% on ≥3 models → {adv_below}/{n_models}")
    print(f"       (mechanisms: {', '.join(sorted(high_adversarial_mechs))})")

    gap_items = set()
    for mn in models:
        for r in sweep_results[mn]:
            gap = abs(r["confidence"] - (1.0 if r["is_correct"] else 0.0))
            if gap > 0.20:
                gap_items.add(r["item_id"])
    c4 = len(gap_items) >= 10
    print(f"  [{'✓' if c4 else '✗'}] C4: ≥10 gap items → {len(gap_items)}")

    model_eces = {}
    for mn in models:
        items = sweep_results[mn]
        bins = defaultdict(list)
        for r in items:
            bins[min(int(r["confidence"] * 5), 4)].append(r)
        ece = sum(
            abs(sum(r["confidence"] for r in bi)/len(bi) - sum(1 for r in bi if r["is_correct"])/len(bi))
            * len(bi) / len(items)
            for bi in bins.values() if bi
        )
        model_eces[mn] = ece

    ece_range = max(model_eces.values()) - min(model_eces.values())
    c5 = ece_range >= 0.03
    print(f"  [{'✓' if c5 else '✗'}] C5: ECE range ≥ 0.03 → {ece_range:.4f}")

    total = sum([c1, c2, c3, c4, c5])
    verdict = "FREEZE ✓" if total >= 4 else "MARGINAL" if total >= 3 else "NEEDS WORK"
    print(f"\n  VERDICT: {total}/5 → {verdict}")

    # ── 5. Audit CSV ──
    audit_rows = []
    for mn in models:
        short = mn.split("/")[-1]
        for r in sweep_results[mn]:
            audit_rows.append({"model": short, **{k: v for k, v in r.items() if k != "would_verify"}})

    audit_df = pd.DataFrame(audit_rows)
    print(f"\nAudit CSV: {len(audit_df)} rows. Export with:")
    print(f"  audit_df.to_csv('metajudge_sweep_audit.csv', index=False)")


In [ ]:
%choose metacog_calibration_v1_batch